In [1]:
# ==============================
# 1. Install Libraries
# ==============================

# !pip install ultralytics

In [2]:
# ==============================
# 2. Import Libraries
# ==============================

from ultralytics import YOLO

import os
import shutil
import random
from pathlib import Path
import yaml

import torch

torch.cuda.empty_cache()

In [3]:
# ==============================
# 3. Dataset Paths
# ==============================

# Real dataset (your current dataset)
REAL_IMAGES = "CroppedDataset/DatasetReal"
REAL_LABELS = "CroppedDataset/labelsReal"

# Synthetic dataset (future Unreal dataset)
SYN_IMAGES = "SyntheticDataset/DatasetSynthetic"
SYN_LABELS = "SyntheticDataset/LabelSynthetic"

In [4]:
# ==============================
# 4. Create YOLO Dataset Split
# ==============================

def split_dataset(images_path, labels_path, output_path):

    images = list(Path(images_path).glob("*.*"))

    random.shuffle(images)

    train_size = int(len(images)*0.7)
    val_size = int(len(images)*0.2)

    datasets = {
        "train": images[:train_size],
        "val": images[train_size:train_size+val_size],
        "test": images[train_size+val_size:]
    }


    for split in datasets:

        os.makedirs(
            f"{output_path}/images/{split}",
            exist_ok=True
        )

        os.makedirs(
            f"{output_path}/labels/{split}",
            exist_ok=True
        )


        for img in datasets[split]:

            label = Path(labels_path) / (img.stem+".txt")


            shutil.copy(
                img,
                f"{output_path}/images/{split}/{img.name}"
            )


            if label.exists():

                shutil.copy(
                    label,
                    f"{output_path}/labels/{split}/{label.name}"
                )

In [5]:
# ==============================
# 5. Prepare Real Dataset
# ==============================

split_dataset(
    REAL_IMAGES,
    REAL_LABELS,
    "YOLO_REAL"
)

In [6]:
# ==============================
# 6. Prepare Synthetic Dataset
# ==============================

# Run after Unreal dataset exists

split_dataset(
    SYN_IMAGES,
    SYN_LABELS,
    "YOLO_SYNTHETIC"
)

In [7]:
# ==============================
# 7. Create YAML File
# ==============================

classes = ["object"]


def create_yaml(path, dataset_name):

    classes = [""]*16

    classes[15] = "pen"


    data = {

        "path": os.path.abspath(dataset_name),

        "train":"images/train",
        "val":"images/val",
        "test":"images/test",

        "names":{
            i:name for i,name in enumerate(classes)
        }

    }


    with open(path,"w") as file:

        yaml.dump(data,file)



create_yaml(
    "real.yaml",
    "YOLO_REAL"
)


create_yaml(
    "synthetic.yaml",
    "YOLO_SYNTHETIC"
)

In [8]:
# ==============================
# 8. Train YOLO Real Dataset
# ==============================

model_real = YOLO("yolov8n.pt")


model_real.train(

    data="real.yaml",

    epochs=100,

    imgsz=416,

    batch=4
)

Ultralytics 8.4.75  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce MX330, 2048MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=real.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([15])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017DA48F6310>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048

In [9]:
# ==============================
# 9. Evaluate Real Dataset
# ==============================

real_results = model_real.val(
    data="real.yaml",
    split="test"
)

real_results

Ultralytics 8.4.75  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce MX330, 2048MiB)
Model summary (fused): 73 layers, 3,008,768 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.40.1 ms, read: 173.8156.0 MB/s, size: 1254.5 KB)
val: Scanning C:\Users\Asus\Documents\André\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\labels\test... 46 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 46/46 234.0it/s 0.2s.4s
val: New cache created: C:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 7.1s/it 21.3s2.4s0
                   all         46         46      0.999          1      0.995      0.799
                   pen         46         46      0.999          1      0.995      0.799
Speed: 8.6ms preprocess, 16.9ms inference, 0.0ms loss, 5.5ms postprocess per image
Results saved to C:\Users\Asus\Document

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([15])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017DD76AA910>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048

In [10]:
# ==============================
# 10. Train YOLO Synthetic Dataset
# ==============================

model_syn = YOLO("yolov8n.pt")


model_syn.train(

    data="synthetic.yaml",

    epochs=100,

    imgsz=416,

    batch=4
)

Ultralytics 8.4.75  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce MX330, 2048MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=synthetic.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspecti

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([15])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017DA5104050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048

In [11]:
synthetic_test_results = model_syn.val(
    data="synthetic.yaml",
    split="test"
)

Ultralytics 8.4.75  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce MX330, 2048MiB)
Model summary (fused): 73 layers, 3,008,768 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.50.2 ms, read: 220.1221.8 MB/s, size: 893.2 KB)
val: Scanning C:\Users\Asus\Documents\André\universidade\Mestrado\projeto_cg\Datasets run\YOLO_SYNTHETIC\labels\test... 8 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8/8 245.1it/s 0.0s
val: New cache created: C:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\YOLO_SYNTHETIC\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 13.8s/it 13.8s
                   all          8          8          1      0.993      0.995      0.733
                   pen          8          8          1      0.993      0.995      0.733
Speed: 29.5ms preprocess, 36.2ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to C:\Users\Asus\Document

In [12]:
# ==============================
# 11. Test Synthetic Model on Real Images
# ==============================

cross_results = model_syn.val(

    data="real.yaml",
    split="test"

)


cross_results

Ultralytics 8.4.75  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce MX330, 2048MiB)
val: Fast image access  (ping: 0.60.4 ms, read: 330.034.0 MB/s, size: 1657.9 KB)
val: Scanning C:\Users\Asus\Documents\André\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\labels\test.cache... 46 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 46/46  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 4.0s/it 12.1s2.7s9
                   all         46         46      0.734      0.848      0.758      0.294
                   pen         46         46      0.734      0.848      0.758      0.294
Speed: 4.1ms preprocess, 8.8ms inference, 0.1ms loss, 7.6ms postprocess per image
Results saved to C:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\runs\detect\val-5


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([15])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017DA4FD2610>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048

In [13]:
# ==============================
# 12. Prediction Example
# ==============================

model_syn.predict(

    source="YOLO_REAL/images/test",

    save=True,

    conf=0.25

)


image 1/46 c:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\images\test\IMG_20260520_172525_1.jpg: 416x416 (no detections), 30.8ms
image 2/46 c:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\images\test\IMG_20260520_172526.jpg: 416x416 1 pen, 30.1ms
image 3/46 c:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\images\test\IMG_20260520_172634.jpg: 416x416 2 pens, 23.1ms
image 4/46 c:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\images\test\IMG_20260520_172703_2.jpg: 416x416 1 pen, 30.8ms
image 5/46 c:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\images\test\IMG_20260520_172706.jpg: 416x416 1 pen, 62.6ms
image 6/46 c:\Users\Asus\Documents\Andr\universidade\Mestrado\projeto_cg\Datasets run\YOLO_REAL\images\test\IMG_20260520_172712_3.jpg: 416x416 1 pen, 33.0ms
image 7/46 c:\Users\Asus\Documents\Andr\universidade

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: '', 1: '', 2: '', 3: '', 4: '', 5: '', 6: '', 7: '', 8: '', 9: '', 10: '', 11: '', 12: '', 13: '', 14: '', 15: 'pen'}
 obb: None
 orig_img: array([[[141, 197, 208],
         [142, 198, 209],
         [143, 199, 210],
         ...,
         [ 71, 199, 120],
         [ 67, 198, 118],
         [ 66, 197, 117]],
 
        [[140, 196, 207],
         [141, 197, 208],
         [142, 198, 209],
         ...,
         [ 72, 200, 121],
         [ 68, 199, 119],
         [ 67, 198, 118]],
 
        [[141, 195, 206],
         [143, 197, 208],
         [144, 198, 209],
         ...,
         [ 72, 200, 123],
         [ 69, 199, 122],
         [ 68, 198, 121]],
 
        ...,
 
        [[ 33, 194, 132],
         [ 21, 180, 118],
         [ 15, 170, 109],
         ...,
         [ 55, 107, 107],
         [ 66, 118, 118],
         [ 71, 123, 123]],
 
  

In [14]:
# ==============================
# 13. Metrics Comparison
# ==============================

results = {

    "Real trained model":
        model_real.metrics,

    "Synthetic trained model":
        model_syn.metrics

}


results

{'Real trained model': ultralytics.utils.metrics.DetMetrics object with attributes:
 
 ap_class_index: array([15])
 box: ultralytics.utils.metrics.Metric object
 confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017DD76AA910>
 curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
 curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
           0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046